# GRASP One-Shot Gene Designer

Put in a **target RNA**. Get back **oligos** that Golden Gate–assemble into the binder gene.

One joint search co-designs:

- **cut sites** — fragment count and length, codon-aligned
- **overhangs** — high-fidelity 4-nt sticky ends, including your destination pair
- **sequence** — codon optimality, cut-site depletion, synthesis heuristics

This path does **not** emit combinatorial library modules or ARELF junctions. For the 42-module library, open `grasp_library_designer.ipynb`.

1. Run **0 · Install** once  
2. Fill the forms and run each cell top → bottom  

Code is hidden by default (Colab Forms). Cell ⋮ → **Form → Hide code** if needed.


In [ ]:
#@title 0 · Install (PyPI) { display-mode: "form" }
#@markdown Installs everything from PyPI. No GitHub token needed. Re-run if imports fail after a runtime restart.

%pip install -q -U --force-reinstall "grasp-library-designer>=0.1.14"
import importlib, sys
for _m in [m for m in list(sys.modules) if m == 'grasp_library' or m.startswith('grasp_library.')]:
    del sys.modules[_m]


import grasp_library
from importlib.metadata import version

print("grasp-library-designer", version("grasp-library-designer"))
print("import ok:", grasp_library.__name__)
from grasp_library import notebook_ui as _ui
print("kazusa_codon_reminder:", hasattr(_ui, "kazusa_codon_reminder"))


In [ ]:
#@title 1 · Settings { display-mode: "form" }
#@markdown Target RNA, host codon table, synthesis limits, assembly enzyme, and destination sticky ends. Run this cell after editing.
#@markdown **Codon tables:** browse [Kazusa CUTG](https://www.kazusa.or.jp/codon/) (search → copy the `species=` accession from the URL).

target_rna = "UUACACGUG" #@param {type:"string"}
organism = "Escherichia coli (Kazusa)" #@param ["Escherichia coli (Kazusa)", "Saccharomyces cerevisiae (Kazusa)", "Homo sapiens (Kazusa)", "Euglena gracilis nuclear (Kazusa)", "Chlamydomonas reinhardtii nuclear (Kazusa)", "Chlamydomonas reinhardtii chloroplast (Kazusa)", "Fetch from Kazusa by species ID", "Upload your own codon table"]
kazusa_species_id = "" #@param {type:"string"}
genetic_code = 1 #@param {type:"integer"}
synthesis_vendor = "Twist · Standard gene guidelines" #@param ["Twist · Express / Low complexity", "Twist · Standard gene guidelines", "Twist · Complex Genes tolerant", "IDT · gBlocks / eBlocks conservative", "Generic · conservative (default)"]
wrap_enzyme = "BsaI (GGTCTC)" #@param ["BsaI (GGTCTC)", "BpiI / BbsI (GAAGAC)", "BsmBI / Esp3I (CGTCTC)"]
#@markdown `wrap_enzyme` is the Type IIS enzyme on every oligo (one-pot assembly).
assembly_enzyme = "BsaI (GGTCTC)" #@param ["BsaI (GGTCTC)", "BpiI / BbsI (GAAGAC)", "BsmBI / Esp3I (CGTCTC)", "GRASP default · BsaI + BpiI + BsmBI", "None (no enzyme filter)"]
#@markdown `assembly_enzyme` plus the blacklist are depleted from the **coding sequence**.
site_blacklist = "SapI, BsaI, BpiI" #@param {type:"string"}
ligation_table = "GRASP Level \u22121 proxy \u00b7 BsaI-HFv2 + T4 \u00b7 37\u219416 \u00b0C cycling (Pryor 2020)" #@param ["GRASP Level \u22121 proxy \u00b7 BsaI-HFv2 + T4 \u00b7 37\u219416 \u00b0C cycling (Pryor 2020)", "GRASP Level 0 proxy \u00b7 BbsI-HF + T4 \u00b7 37\u219416 \u00b0C cycling (Pryor 2020)", "T4 ligase only \u00b7 18 h \u00b7 25 \u00b0C (Potapov 2018; validated cycling proxy)", "T4 ligase only \u00b7 1 h \u00b7 25 \u00b0C (Potapov 2018)", "T4 ligase only \u00b7 1 h \u00b7 37 \u00b0C (Potapov 2018)", "T4 ligase only \u00b7 18 h \u00b7 37 \u00b0C (Potapov 2018)"]
destination_5prime_overhang = "CTCA" #@param {type:"string"}
destination_3prime_overhang = "CTCG" #@param {type:"string"}
#@markdown Destination sticky ends (physical, 5\u2032\u21923\u2032) for cloning the assembled gene. They are adapters, not part of the protein.
n_fragments = 0 #@param {type:"integer"}
#@markdown `n_fragments = 0` chooses a count from the vendor oligo-length cap.
optimize_depth = 400 #@param {type:"integer"}

from grasp_library import build_default_config, materialize_project
from grasp_library.colab_forms import apply_form_settings
from grasp_library import notebook_ui as ui

PROJECT_DIR = materialize_project()
INPUT_DIR = PROJECT_DIR / "input"
OUTPUT_ROOT = PROJECT_DIR / "output" / "oneshot"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

CONFIG = build_default_config(INPUT_DIR)
CONFIG["project_name"] = "GRASP_oneshot_colab"

if hasattr(ui, "kazusa_codon_reminder"):
    ui.kazusa_codon_reminder()
else:
    ui.note(
        "Package is outdated (missing Kazusa helper). "
        "Re-run <b>0 · Install</b>, then Runtime → Restart session, then Settings again."
    )

applied = apply_form_settings(
    CONFIG,
    organism=organism,
    genetic_code=int(genetic_code),
    target_rna=target_rna,
    synthesis_vendor=synthesis_vendor,
    assembly_enzyme=assembly_enzyme,
    site_blacklist=site_blacklist,
    ligation_table=ligation_table,
    optimize_depth=int(optimize_depth),
    kazusa_species_id=kazusa_species_id,
    redesign_plasmid_overhangs=False,
    redesign_level0_junctions=False,
)
CONFIG = applied["config"]
CODON_DATA = applied["codon_data"]
ONESHOT = None
ORGANISM_LABEL = applied.get("meta", {}).get("organism", organism)
CONFIG["oneshot"] = {
    "n_fragments": None if int(n_fragments) <= 0 else int(n_fragments),
    "destination_5prime_overhang": destination_5prime_overhang.strip().upper(),
    "destination_3prime_overhang": destination_3prime_overhang.strip().upper(),
    "wrap_enzyme": wrap_enzyme,
}

ui.status(
    f"Target <b>{CONFIG['target_rna']}</b> · organism <b>{ORGANISM_LABEL}</b> · "
    f"genetic code <b>{CONFIG['genetic_code']}</b> · "
    f"depth <b>{CONFIG['optimizer']['iterations_per_part']:,}</b> · "
    f"vendor <b>{CONFIG['synthesis_vendor']}</b><br/>"
    f"Wrap <b>{wrap_enzyme}</b> · fragments <b>{'auto' if int(n_fragments) <= 0 else n_fragments}</b> · "
    f"destination 5\u2032/3\u2032 <code>{destination_5prime_overhang}/{destination_3prime_overhang}</code><br/>"
    f"Ligation model: <b>{CONFIG['ligation']['table_name']}</b> · Project → <code>{PROJECT_DIR}</code>"
)


In [ ]:
#@title 2 · Preview binder protein { display-mode: "form" }

from grasp_library import describe_binder
from grasp_library import notebook_ui as ui

info = describe_binder(CONFIG["target_rna"])
ui.status(
    f"<b>{info['target_rna']}</b> · PPR <code>{info['ppr_code']}</code> · "
    f"<b>{info['aa_length']}</b> aa · <b>{info['cds_length']}</b> nt · "
    f"architecture <b>{len(CONFIG['target_rna'])}S</b> · continuous gene (no library modules)"
)
print(info["aa_sequence"])


In [ ]:
#@title 3 · Co-design oligos { display-mode: "form" }
#@markdown Jointly chooses cuts, overhangs, and sequence. Files appear under `grasp_library_project/output/oneshot/`.

RUN_ONESHOT = True #@param {type:"boolean"}
SEED = 42 #@param {type:"integer"}

import random
import numpy as np
from IPython.display import display
from grasp_library import run_oneshot_design, sanitize_rna_name
from grasp_library import notebook_ui as ui

random.seed(int(SEED))
np.random.seed(int(SEED))

ONESHOT = None
if not RUN_ONESHOT:
    ui.note("RUN_ONESHOT is off.")
elif not CODON_DATA:
    ui.note("Run the Settings cell first.")
else:
    rna = sanitize_rna_name(CONFIG["target_rna"])
    out_dir = OUTPUT_ROOT / rna
    ONESHOT = run_oneshot_design(
        target_rna=CONFIG["target_rna"],
        codon_data=CODON_DATA,
        config=CONFIG,
        output_dir=out_dir,
        seed=int(SEED),
        log=print,
    )
    display(ONESHOT["assembly_plan"][
        [c for c in [
            "fragment_id", "assembly_order", "aa_length",
            "oh5_coding_site_5to3", "oh3_coding_site_5to3",
            "ligation_fidelity_set",
        ] if c in ONESHOT["assembly_plan"].columns]
    ])
    cols = [c for c in [
        "order_fragment_id", "assembly_order", "oligo_length", "oligo_gc",
        "qc_status", "hard_constraints_passed", "order_sequence_5to3",
    ] if c in ONESHOT["oligos"].columns]
    display(ONESHOT["oligos"][cols])
    asm = ONESHOT["assembled"]
    ui.status(
        f"Translation verified: <b>{asm['translation_verified']}</b> · "
        f"codon table OK: <b>{asm.get('codon_table_ok')}</b> · "
        f"<b>{asm.get('n_fragments')}</b> oligos · "
        f"ligation fidelity <b>{asm.get('ligation_fidelity'):.4f}</b> · "
        f"order file → <code>{ONESHOT['order_csv']}</code>"
    )


In [ ]:
#@title 4 \u00b7 Export Excel { display-mode: "form" }

import pandas as pd
from grasp_library import notebook_ui as ui, write_qc_sheet

if ONESHOT is None:
    ui.note("Run Design first.")
else:
    out = ONESHOT["output_dir"]
    xlsx = out / f"oneshot_{ONESHOT['target_rna']}.xlsx"
    with pd.ExcelWriter(xlsx) as writer:
        pd.DataFrame([ONESHOT["binder"]]).to_excel(writer, sheet_name="binder", index=False)
        ONESHOT["assembly_plan"].to_excel(writer, sheet_name="assembly_plan", index=False)
        ONESHOT["oligos"].to_excel(writer, sheet_name="oligos", index=False)
        pd.DataFrame([ONESHOT["assembled"]]).to_excel(writer, sheet_name="assembled_gene", index=False)
        pd.DataFrame([ONESHOT["summary"]]).to_excel(writer, sheet_name="summary", index=False)
        write_qc_sheet(writer, ONESHOT["oligos"])
    ui.status(f"Wrote <code>{xlsx}</code>")
    if "google.colab" in __import__("sys").modules:
        from google.colab import files
        files.download(str(xlsx))
    for p in sorted(out.glob("*")):
        if p.is_file():
            print(p.name)


## Notes

| Step | What happens |
|---|---|
| Install | `pip install grasp-library-designer` from PyPI |
| RNA → protein | PPR code builds the continuous binder (no library parts) |
| Joint design | Beam-search cut sites and synonymous overhangs for ligation fidelity, then recode the CDS for codon optimality, blacklist depletion, and synthesis |
| Oligos | Each fragment is wrapped with inward-facing Type IIS sites and your destination sticky ends |
| Export | Orderable oligo table, assembly plan, and assembled gene FASTA |

Ligation scores are model surrogates, not cloning guarantees. Vendor QC is a transparent heuristic (`vendor_acceptance_confirmed=False`).

For the **42-module combinatorial library**, open `grasp_library_designer.ipynb`.

Package: https://pypi.org/project/grasp-library-designer/
